# Direct neural-network baselines

This notebook runs the paper's fully connected ReLU baseline for Scenario A or B. Each repetition uses three independently simulated particle systems: paths for training, an empirical-measure system for validation, and paths for final testing. The test system is evaluated exactly once.

## 1. Protocol

The randomized search considers 1--4 hidden layers, widths 16--128, and learning rates $10^{-4}$--$10^{-2}$. Adam updates are followed by coefficient projection. The 20-epoch search selects one candidate using accuracy on the complete empirical-measure validation system. That candidate is reinitialized and fully trained exactly once. There is no runner-up, loss-based rescue, seed replacement, or selective rerun: optimization failures and class collapses remain in the reported Monte Carlo sample.

In [ ]:
from pathlib import Path
import csv, sys
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from mkv_classification.config import NeuralBaselineConfig, load_config, neural_baseline_profile
from mkv_classification.experiments_nn import run_neural_baseline
SCENARIO='A'  # Use 'A' or 'B'.
PROFILE='smoke'  # Change only this line to 'paper'.
config_path=ROOT/'configs'/f'nn_scenario_{SCENARIO.lower()}_paper.json'
config=(load_config(config_path,NeuralBaselineConfig) if PROFILE=='paper'
        else neural_baseline_profile(PROFILE,SCENARIO))
output_dir=ROOT/'results'/f'nn_{SCENARIO}_{PROFILE}_empirical_measure_single_finalist'
config

## 2. Run or resume

In [ ]:
rows=run_neural_baseline(config,output_dir)
print(f'{len(rows)} rows available in {output_dir}')

## 3. Paper-ready summary

In [ ]:
with (output_dir/'summary.csv').open(newline='',encoding='utf-8') as handle:
    summary=list(csv.DictReader(handle))
columns=['scenario','regime','M','N_train_per_class','N_validation_per_class','repetitions','direct_nn_accuracy_mean','direct_nn_accuracy_sample_standard_deviation','direct_nn_accuracy_median','direct_nn_accuracy_minimum','class_collapse_count','class_collapse_rate','chance_or_worse_count','chance_or_worse_rate']
print(' | '.join(columns)); print(' | '.join(['---']*len(columns)))
for row in summary: print(' | '.join(row[column] for column in columns))

## 4. Reporting boundary

Run this notebook separately for `SCENARIO='A'` and `SCENARIO='B'`. Here, `N_train_per_class=N_validation_per_class`; training paths and empirical-measure validation particles come from two independent systems and are never pooled. All pre-specified repetitions are reported, including collapsed and chance-level runs. Smoke outputs are integrity checks. The direct NN is a benchmark rather than part of the plug-in estimator, and its final test system is never used for hyperparameter selection.